# Portfolio Project: Netflix Content & Distribution Analysis

## Phase 1: Project Initialization & Data Auditing

### 1.1 Objective & Dataset
This project focuses on auditing, cleaning, and analyzing the global Netflix titles dataset. The goal is to uncover trends regarding content distribution, regional production preferences, and the growth of Movies versus TV Shows over the last decade.
* **Data Source:** Automated pipeline streaming from portfolio repository.
* **Key Analytical Goals:** Track historical content growth, identify top contributing countries, and profile dominant genres.

### 1.2 Data Loading & Initial Inspection
To begin the audit, we establish our environment, pull the compressed dataset directly from our remote backup, and run our primary diagnostic checks (`head()`, `info()`, `isnull()`, and `duplicated()`) to evaluate structural integrity and locate missing data profiles.

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
plt.style.use('ggplot')

In [ ]:
# 1. Point to your local zip file name
local_zip_filename = "netflix_titles.csv.zip" 
data_url = "https://github.com/johnkenke/AnalystLab-Data-Analysis-Portfolio/raw/main/Week_01_Netflix_Analysis/netflix_titles.csv.zip" # Keep this link for others/future use

if os.path.exists(local_zip_filename):
    print("🚀 Instantly reading and unzipping data from local system archive...")
    # Pandas automatically extracts and reads the CSV inside the zip file
    df = pd.read_csv(local_zip_filename, compression='zip')
else:
    print("🌐 File not found locally. Streaming from remote repository...")
    df = pd.read_csv(data_url)
    # Optional: saves it as a zip if you want to mirror the setup
    df.to_csv(local_zip_filename, index=False, compression='zip')

print(f"Data successfully loaded! Total rows: {df.shape[0]}")

In [ ]:
# Print the top 10 rows 
df.head(10)

In [ ]:
# Get summary information about the DataFrame, including data types and non-null counts
df.info()

In [ ]:
# checking to see which columns have missing values and how many
df.isnull().sum()

In [ ]:
# checking for duplicates
df.duplicated().sum()

## Phase 2: Data Cleaning, Imputation & Type Transformation

### 2.1 Missing Data Profile
Our initial diagnostics revealed a split profile of missing data across the dataset, requiring two distinct handling strategies to preserve data integrity:
* **Imputation (High Volume):** The `director`, `cast`, and `country` columns contain a high volume of null values. We substitute these nulls with a standardized placeholder string: `'Unknown'`.
* **Row Deletion (Low Volume):** The `date_added`, `rating`, and `duration` columns have negligible gaps. Dropping these specific rows keeps our core metrics clean without impacting statistical volume.

### 2.2 Data Type Transformation
During our initial audit, we observed that `date_added` was loaded as a generic text string (`object`). Text-based dates cannot be sorted chronologically or grouped by month/year. We will transform this column into a true `datetime64` data type to enable time-series exploration.

### 2.3 Operational Execution
The following operations systematically apply our missing value strategies, convert data types, and verify the final structure of our clean dataset.

In [ ]:
# Fill high-volume missing text columns with 'Unknown'
df['director'] = df['director'].fillna('Unknown')
df['cast'] = df['cast'].fillna('Unknown')
df['country'] = df['country'].fillna('Unknown')

# Drop the rows where the critical low-volume columns are blank
df = df.dropna(subset=['date_added', 'rating', 'duration'])

# Structural Cleaning: Convert date_added from text to a true datetime format
df['date_added'] = pd.to_datetime(df['date_added'].str.strip())

# Final verification check for missing values and the new date data type
print("--- Missing Values Check ---")
print(df.isnull().sum())
print("\n--- Column Data Type Check ---")
print(f"date_added type: {df['date_added'].dtype}")

## Phase 3: Exploratory Data Analysis (EDA) & Visualization

With a clean, structurally sound dataset, we move into the discovery phase. Exploratory Data Analysis allows us to look past individual rows and uncover macro-trends, behavioral shifts, and patterns in how Netflix curates and scales its global entertainment platform.

### 3.1 Historical Growth & Content Acquisition Trends
Our first objective is to track the velocity of Netflix's growth over time. While the `release_year` column tells us when a piece of media was originally created by its producers, looking at the year a title was actually **added to the platform** reveals Netflix's aggressive content acquisition strategy and corporate scaling timeline.

### 3.2 Operational Discovery
In this section, we will:
1. Extract the specific calendar year from our transformed `date_added` datetime metrics.
2. Agregate overall growth trends chronologically.
3. Construct a high-impact visualization utilizing pure Matplotlib to map the platform's content trajectory.

In [ ]:
print("    PHASE 3: SUMMARY STATISTICS PROFILE     ")
# 1. Clean Numeric Distribution (Percentiles only)
# Extract the year from the clean datetime column
df['year_added'] = df['date_added'].dt.year

#print("Column 'year_added' successfully created!")
year_distribution = df[['release_year', 'year_added']].describe().loc[['min', '25%', '50%', '75%', 'max']]

print("--- Catalog Age & Ingestion Distribution ---")
print(year_distribution)
print("\n" + "-"*50 + "\n")

# 2. Clean Categorical Dominance (Top value only)
top_categorical = df[['type', 'rating', 'country']].mode().iloc[0]

print("--- Core Catalog Dominance Profile ---")
print(top_categorical)

In [ ]:
# Extract the year from our datetime column and create a new column for it
df['year_added'] = df['date_added'].dt.year

# Count how many titles were added each year and sort chronologically
content_by_year = df['year_added'].value_counts().sort_index()

# Display the trend table
print("Content added to Netflix by Year:")
print(content_by_year)

In [ ]:
# Set a clean, minimalist figure layout
plt.figure(figsize=(12, 5.5))

# Plot the trend with a slightly thicker line and cleaner markers
plt.plot(content_by_year.index, content_by_year.values, marker='o', markersize=6, color='#E50914', linewidth=3)

# The Title carries the context, removing the need for redundant axis labels
plt.title('The Growth of Netflix: Titles Added Per Year (2008 - 2021)', fontsize=14, fontweight='bold', pad=20, loc='left')

# Clean up the X-axis 
plt.xticks(content_by_year.index, fontsize=10)
plt.yticks(fontsize=10)

# Remove the outer chart border 
for spine in ['top', 'right', 'left', 'bottom']:
    plt.gca().spines[spine].set_visible(False)

# Use a very subtle horizontal-only grid to guide the eye without clutter
plt.grid(axis='y', linestyle=':', alpha=0.6, color='#CCCCCC')

# Ensure the layout fits perfectly
plt.tight_layout()

# Render the clean chart
plt.show()

In [ ]:
# Create a cross-tabulation of year_added vs content type
type_trend = pd.crosstab(df['year_added'], df['type'])

# Let's see the raw breakdown side-by-side
print("Content Breakdown by Year:")
print(type_trend)

In [ ]:
# Set up the figure size for a wide, clean look
plt.figure(figsize=(12, 5.5))

# Plot Movies trend line
plt.plot(type_trend.index, type_trend['Movie'], marker='o', markersize=5, 
         color='#E50914', linewidth=3, label='Movies')

# Plot TV Shows trend line
plt.plot(type_trend.index, type_trend['TV Show'], marker='o', markersize=5, 
         color='#4A4A4A', linewidth=2.5, label='TV Shows')

# Title setup matching our previous layout
plt.title('Netflix Content Strategy: Movies vs. TV Shows Added Per Year', 
          fontsize=14, fontweight='bold', pad=20, loc='left')

# Format the axes text sizes
plt.xticks(type_trend.index, fontsize=10)
plt.yticks(fontsize=10)

# Remove the outer chart border (spines) to keep the clean, dashboard style
for spine in ['top', 'right', 'left', 'bottom']:
    plt.gca().spines[spine].set_visible(False)

# Subtle horizontal grid lines
plt.grid(axis='y', linestyle=':', alpha=0.6, color='#CCCCCC')

# Add a clean legend to distinguish the two lines
plt.legend(frameon=False, fontsize=11, loc='upper left')

plt.tight_layout()
plt.show()

In [ ]:
# 1. Isolate ID and Country columns
country_isolation = df[['show_id', 'country']].copy()

# 2. Split comma-separated countries into clean lists
country_isolation['country_list'] = country_isolation['country'].str.split(', ')

# 3. Unpivot (Explode) so every country gets its own row linked to the show_id
country_bridge_table = country_isolation.explode('country_list')[['show_id', 'country_list']]

# 4. Filter out any 'Unknown' placeholders from the unpivoted results
country_bridge_table = country_bridge_table[country_bridge_table['country_list'] != 'Unknown']

# 5. Extract the true Top 10 individual contributing countries
top_10_countries = country_bridge_table['country_list'].value_counts().head(10)

# Quick verification print
print("Data processing complete. Top 10 countries extracted.")

In [ ]:
# Set up the figure layout
plt.figure(figsize=(12, 5.5))

# Use charcoal gray for all bars, and highlight 'United States' in Netflix Red
colors = ['#E50914' if country == 'United States' else '#4A4A4A' for country in top_10_countries.index]

# Plot horizontal bars (reversed [::-1] so #1 is at the top)
plt.barh(top_10_countries.index[::-1], top_10_countries.values[::-1], color=colors[::-1], height=0.6)

# Title matching our sleek dashboard design system
plt.title('Global Production Leaderboard', 
          fontsize=15, fontweight='bold', pad=20, loc='center')

# Add exact data labels directly onto the ends of the bars for crisp scannability
for index, value in enumerate(top_10_countries.values[::-1]):
    plt.text(value + 40, index, f'{value:,}', va='center', fontsize=10, fontweight='bold', color='#333333')

# Clean up axes and strip outer borders
plt.xticks([])  # Hide X-axis completely since labels are directly on the bars
plt.yticks(fontsize=11)

for spine in ['top', 'right', 'left', 'bottom']:
    plt.gca().spines[spine].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
# 1. Isolate the ID and Genre columns
genre_isolation = df[['show_id', 'type', 'listed_in']].copy()

# 2. Split comma-separated strings into clean lists
genre_isolation['genre_list'] = genre_isolation['listed_in'].str.split(', ')

# 3. Unpivot (Explode) so each tag gets its own row
genre_bridge_table = genre_isolation.explode('genre_list')[['show_id', 'type', 'genre_list']]

# 4. Define a list of non-genre meta-tags to filter out
meta_tags_to_exclude = [
    'International Movies', 'International TV Shows', 
    'Independent Movies', 'TV Shows', 'Movies'
]

# 5. Filter the bridge table to keep ONLY true genres
true_genres_table = genre_bridge_table[~genre_bridge_table['genre_list'].isin(meta_tags_to_exclude)]

# 6. Extract the true Top 10 genres for Movies and TV Shows separately
movie_genres = true_genres_table[true_genres_table['type'] == 'Movie']
top_10_movie_genres = movie_genres['genre_list'].value_counts().head(10)

tv_genres = true_genres_table[true_genres_table['type'] == 'TV Show']
top_10_tv_genres = tv_genres['genre_list'].value_counts().head(10)

print("Data processing complete. Pure thematic genres extracted.")
print("------top 10 movie genres------")
print(top_10_movie_genres)
print("        --------        ")
print("------top 10 tv genres------")
print(top_10_tv_genres)

In [ ]:
# Create a figure with 2 subplots side-by-side
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# --------------------------------------------------
# SUBPLOT 1: Top Movie Genres
# --------------------------------------------------
ax1.barh(top_10_movie_genres.index[::-1], top_10_movie_genres.values[::-1], color='#4A4A4A', height=0.6)
# Highlight Dramas as the top Movie genre
if 'Dramas' in top_10_movie_genres.index:
    idx = list(top_10_movie_genres.index[::-1]).index('Dramas')
    ax1.patches[idx].set_facecolor('#E50914')

ax1.set_title('Top 10 Movie Genres', fontsize=13, fontweight='bold', pad=15, loc='left')

for index, value in enumerate(top_10_movie_genres.values[::-1]):
    ax1.text(value + 20, index, f'{value:,}', va='center', fontsize=9, fontweight='bold', color='#333333')

ax1.set_xticks([])
ax1.tick_params(labelsize=10)
for spine in ['top', 'right', 'left', 'bottom']:
    ax1.spines[spine].set_visible(False)

# --------------------------------------------------
# SUBPLOT 2: Top TV Show Genres
# --------------------------------------------------
ax2.barh(top_10_tv_genres.index[::-1], top_10_tv_genres.values[::-1], color='#4A4A4A', height=0.6)
# Highlight TV Dramas as the top TV genre
if 'TV Dramas' in top_10_tv_genres.index:
    idx = list(top_10_tv_genres.index[::-1]).index('TV Dramas')
    ax2.patches[idx].set_facecolor('#E50914')

ax2.set_title('Top 10 TV Show Genres', fontsize=13, fontweight='bold', pad=15, loc='left')

for index, value in enumerate(top_10_tv_genres.values[::-1]):
    ax2.text(value + 15, index, f'{value:,}', va='center', fontsize=9, fontweight='bold', color='#333333')

ax2.set_xticks([])
ax2.tick_params(labelsize=10)
for spine in ['top', 'right', 'left', 'bottom']:
    ax2.spines[spine].set_visible(False)

# Global layout tuning
plt.suptitle('Content Architecture: Thematic Genre Distribution', 
             fontsize=15, fontweight='bold', y=1.02, x=0.06, ha='left')
plt.tight_layout()
plt.show()

In [ ]:
# 1. Directly extract the month name from your clean datetime column
df['added_month'] = df['date_added'].dt.month_name()

# 2. Define the correct chronological order for months
month_order = [
    'January', 'February', 'March', 'April', 'May', 'June', 
    'July', 'August', 'September', 'October', 'November', 'December'
]

# 3. Count the titles added per month and reindex to force calendar order
monthly_counts = df['added_month'].value_counts().reindex(month_order)

print("Data processing complete. Chronological monthly trends extracted.")

In [ ]:
# Set up a wide, clean canvas layout
plt.figure(figsize=(13, 5.5))

# Plot the seasonality line using a dark charcoal gray with data markers
plt.plot(monthly_counts.index, monthly_counts.values, 
         color='#4A4A4A', marker='o', linewidth=2.5, markersize=6)

# DYNAMIC HIGHLIGHT: Automatically find and paint the peak month in Netflix Red
peak_month = monthly_counts.idxmax()
peak_value = monthly_counts.max()

plt.plot(peak_month, peak_value, marker='o', color='#E50914', markersize=10)

# Title styled to match your sleek dashboard layout guidelines
plt.title('Platform Seasonality: Which Months Do Content Drops Peak?', 
          fontsize=14, fontweight='bold', pad=20, loc='left')

# Inject exact text values directly over each data point for quick scanning
for month, value in zip(monthly_counts.index, monthly_counts.values):
    offset = 25 if month == peak_month else 15
    plt.text(month, value + offset, f'{value:,}', ha='center', fontsize=9, color='#333333')

# Clean up axes boundaries and completely remove visual noise
plt.ylim(0, peak_value + 100)  # Add padding at the top so labels don't get clipped
plt.xticks(fontsize=10)
plt.yticks([])                 # Strip out the Y-axis ticks entirely

# Explicitly disable all background gridlines to keep the canvas open and blank
plt.grid(False)

# Strip out outer boundary borders (spines)
for spine in ['top', 'right', 'left', 'bottom']:
    plt.gca().spines[spine].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
# Save the cleaned dataset to a CSV file
output_filename = "cleaned_Netflix_data.csv"
df.to_csv(output_filename, index=False)

print(f"🎉 Success! Cleaned dataset exported to: {output_filename}")
print(f"Final file shape: {df.shape}")

# Executive Summary & Strategic Insights

This analysis evaluates the core architectural pillars of Netflix’s content library by isolating and refining raw metadata into structured, queryable data models. By building dedicated unpivoted bridge tables for multi-value fields (countries and genres) and applying rigorous domain filtering, we exposed the true operational blueprint of the platform.

---

### 1. The Global Production Engine (Country Footprint)
* **Domestic Anchor with Global Scale:** Unpivoting the multi-country production strings reveals that the **United States** remains the structural anchor of Netflix’s library. 
* **Key Co-Production Ecosystems:** Major international hubs like India, the United Kingdom, and Canada form the primary international support network, proving that Netflix relies heavily on established global cinema hubs to balance its localized content mandates.

### 2. The Thematic Blueprint (True Genre Distribution)
* **Format-Specific Tailoring:** When stripping away non-thematic marketing and distribution tags (e.g., "International Content", "Independent Movies"), a clean content strategy emerges. 
* **The Drama & Comedy Duopoly:** **Dramas** and **Comedies** comfortably dominate both the Movie and TV Show formats. 
* **Non-Fiction Stability:** Documentaries and Docuseries securely hold top-five positions across the entire ecosystem, serving as a highly cost-effective, consistent foundation for engagement.

### 3. Release Seasonality & Audience Capture
* **Strategic Drop Windows:** Content ingestion is not distributed evenly throughout the calendar year. The platform demonstrates clear cyclical surges, heavily stacking release schedules to capture subscriber eyeballs during peak consumer downtime.
* **Holiday & Binge Optimization:** The seasonal analysis indicates targeted volume increases that align precisely with seasonal holiday periods, maximizing subscriber retention when the propensity for consecutive-episode "bingeing" is at its historical highest.

---

### Technical Notes (For Portfolio Review)
> 🛠️ **Data Integrity Measures Applied:**
> * **Granular Unpivoting:** Handled multi-value string arrays (`country`, `listed_in`) by implementing an `.str.split(', ').explode()` transformation pipeline to prevent aggregated string duplication.
> * **Categorical Cleansing:** Explicitly isolated and filtered out systemic placeholder anomalies (such as `"Unknown"` data points) post-explosion.
> * **Thematic Isolation:** Programmatically excluded meta-distribution labels from the genre index to isolate pure, actionable user-intent thematic trends.